In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import re
import scipy.optimize as opt
from tqdm import tqdm
import sympy as sp
from sympy.parsing.sympy_parser import parse_expr
from sympy import lambdify
from global_parameter import *
import torch
from torch import nn, optim
from torch.autograd.functional  import hessian 

from torch import log10
from torch import exp

from scipy import stats
from rpy2.robjects import r, FloatVector

In [3]:
filename = 'pareto_low_adv_refit.csv'  
t_eq=pd.read_csv('/data/zj448/SR/Ultimate_paper/pareto_archive/'+filename)

df_full = pd.read_csv('SMBH_Data_03_06_24.csv',header=1)

paras=low_scatter_para_std

In [4]:
booleans=['ETG','Bar','Disk','Ring','Core','Multiple','Compactness','AGN','Pseudobulge','BCG','cD']
for b in booleans:
    df_full[b+'_std']=0
    
df=df_full[low_scatter_para_std].dropna(axis='index',how='any').copy()

In [5]:
t_eq

,complexity,loss,score,equation,sympy_format,lambda_format,number_constants,variables,number_variables,unique_number_variables,evolutions,iterations,fitting_format,num_fitting_variables,initial_constant_guess,LLL,intrinsic_scatter,refit_equation,refit_wrmse
0,1,4.906695,0.000000,x48,x48,PySRFunction(X=>x48),0,{'x48'},1,1,0,0,x48,0,[],-2684.924248,2.502863,x48,4.906695
1,15,0.088578,0.044147,((x12 - (exp(log10(exp(0.04766298070500929) / ...,x12 - (1.048817124146649/(x19 + 0.047662980705...,PySRFunction(X=>x12 - (1.048817124146649/(x19 ...,3,"{'x19', 'x29', 'x12'}",3,3,0,0,x12 - log10(x29) - exp(log10(p[1]/(x19 + p[0])...,3,"['0.04766298070500929', '1.048817124146649', '...",NaN,NaN,x12 - log10(x29) - exp(log10(-0.12211717766330...,NaN
2,16,0.088371,0.002337,((x12 - (exp(log10(exp(exp(-1.0423515630962228...,x12 - (exp(0.35262448748241617/x12)/x19)**(1/l...,PySRFunction(X=>x12 - (exp(0.35262448748241617...,2,"{'x19', 'x29', 'x12'}",4,3,0,0,x12 - log10(x29) - exp(log10(exp(p[0]/x12)/x19...,2,"['0.35262448748241617', '1.29244433001088']",-76.179376,0.424280,x12 - log10(x29) - exp(log10(exp(-892.51893396...,0.107443
3,17,0.086897,0.016823,((x12 - (exp(log10(exp(x29 / x12) / (x19 + 0.0...,x12 - (exp(x29/x12)/(x19 + 0.04766298070500929...,PySRFunction(X=>x12 - (exp(x29/x12)/(x19 + 0.0...,2,"{'x29', 'x12', 'x19'}",5,3,0,0,x12 - log10(x29) - exp(log10(exp(x29/x12)/(x19...,2,"['0.04766298070500929', '1.29244433001088']",-70.893942,0.421744,x12 - log10(x29) - exp(log10(exp(x29/x12)/(x19...,0.101308
4,19,0.086145,0.004346,((x12 - (exp(log10(exp(x29 / x12) / ((x19 - x1...,x12 - (exp(x29/x12)/(-x14 + x19 + 0.0476629807...,PySRFunction(X=>x12 - (exp(x29/x12)/(-x14 + x1...,2,"{'x29', 'x14', 'x12', 'x19'}",6,4,0,0,x12 - log10(x29) - exp(log10(exp(x29/x12)/(-x1...,2,"['0.04766298070500929', '1.29244433001088']",-74.219419,0.422749,x12 - log10(x29) - exp(log10(exp(x29/x12)/(-x1...,0.104474
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
786,13,0.015437,0.052914,((((x15 * 3.0860786087580303) - -2.72029684243...,3.0860786087580303*x15 + 2.835860812650755 - 2...,PySRFunction(X=>3.0860786087580303*x15 + 2.835...,3,"{'x26', 'x15', 'x29'}",4,3,4,2969,p[0]*x15 + p[1] - p[2]*x29/x26,3,"['3.08607860875803', '2.83586081265075', '2.0']",-6.342393,0.129297,2.214948689657893*x15 + 5.318061088249941 - 2....,0.012861
787,11,0.013242,0.190891,((x15 * (3.4412858435521843 - (x29 / (x26 + 0....,x15*(-x29/(x26 + 0.012180693250749317) + 3.441...,PySRFunction(X=>x15*(-x29/(x26 + 0.01218069325...,3,"{'x26', 'x15', 'x29'}",3,3,4,3493,x15*(-x29/(x26 + p[0]) + p[1]) + p[2],3,"['0.0121806932507493', '3.44128584355218', '2....",-2.118180,0.142504,x15*(-x29/(x26 + -0.30270296186787016) + 3.208...,0.015038
788,25,0.012863,0.002140,((((x15 * (3.4412858435521843 - (x29 / x26))) ...,x15*(3.4412858435521843 - x29/x26) + log(exp(1...,PySRFunction(X=>x15*(3.4412858435521843 - x29/...,6,"{'x0', 'x26', 'x15', 'x29'}",5,4,4,3494,x15*(p[2] - x29/x26) + log10(exp(p[1]*x0*exp(-...,5,"['0.8673260387054437', '1.613264904706933', '3...",-1.763859,0.127957,x15*(3.5326679489905857 - x29/x26) + log10(exp...,0.014635
789,22,0.011836,0.014948,(((x15 * ((3.5011432103496922 - (log10(((x46 -...,x15*(3.5011432103496922 - log(1.63819164201210...,PySRFunction(X=>x15*(3.5011432103496922 - log(...,6,"{'x26', 'x15', 'x46', 'x54', 'x29'}",5,5,4,3559,x15*(p[3] - log10(p[1]/(p[0]*x46 + p[5])**p[2]...,6,"['0.39262958179688978', '1.6381916420121015', ...",-2.134484,0.138010,x15*(3.6611160870021457 - log10(23186.27872631...,0.014096


In [6]:
t_eq['final_equation']=pd.Series(dtype='object')
t_eq['logdet']=0.
t_eq['sqrt_cov']=pd.Series(dtype='object')
t_eq['final_loss']=0.
t_eq['optimized_params']=pd.Series(dtype='object')
t_eq['N_para']=0.
t_eq['residual_array']=pd.Series(dtype='object')
t_eq['ad_p_value_array']=pd.Series(dtype='object')
t_eq['final_BIC']=0.
t_eq['final_BIC_i']=0.


# Load R's stats package
r('library(goftest)')

number_matching_pattern = r"(?<![a-zA-Z0-9_.])[+-]?(\d+\.\d+|\.\d+|\d+\.|\d+)(?:[eE][-+]?\d+)?"


M_BH = torch.tensor(df['M_BH'].values, dtype=torch.float64)
M_BH_std_sym = torch.tensor(df['M_BH_std_sym'].values, dtype=torch.float64)

def lr_lambda(epoch):
    start_decay_epoch = 5000 
    total_decay_epochs = 10000 - start_decay_epoch 
    if epoch < start_decay_epoch:
        return 1.0
    else:
        decay_ratio = (epoch - start_decay_epoch) / total_decay_epochs 
        return max(0.0, 1.0 - decay_ratio)
    
for row in tqdm(t_eq[319:320].iterrows(),total=len(t_eq)):
    
    equation = row[1]['refit_equation']

    # Find all unique matches first to avoid duplicates and incorrect indexing
    constants = list(set(re.findall(number_matching_pattern, equation)))
    # Sort constants by their length in descending order to replace longer numbers first, preventing partial replacement issues
    constants.sort(key=len, reverse=True)

    def replace_with_variable(match):
        # Find the matched number in the constants list and get its index
        number = match.group(1)
        index = constants.index(number)

        # Check if the match includes a minus sign
        if match.group(0).startswith('-'):
            sign = '-'
        else:
            sign = ''
        return f'{sign}self.p{index}'

    equation = re.sub(number_matching_pattern, replace_with_variable, equation)

    # get x_indexs
    x_indexs = re.findall(r'x(\d+)', equation)
    x_indexs = list(dict.fromkeys(x_indexs))
    x_indexs = [int(x_index) for x_index in x_indexs]

    # construct tensors for initial position of x
    x=[]
    x_stds=[]
    for x_index in x_indexs:
        x.append(torch.tensor(df[paras[x_index]].values, dtype=torch.float64))
        x_stds.append(torch.tensor(df[paras[x_index]+'_std'].values, dtype=torch.float64))

    # Define the model
    class Model(nn.Module):
        
        def __init__(self):
            # assign initial values to parameters self.p[i], self.x[i] and self.intrinsic_scatter
            super(Model, self).__init__()
            for i in range(len(constants)):
                setattr(self, f'p{i}', nn.Parameter(torch.tensor(float(constants[i]), dtype=torch.float64)))
            for i,x_index in enumerate(x_indexs):
                if x_stds[i].sum() != 0:
                    setattr(self, f'x{x_index}', nn.Parameter(x[i].clone()))
                else:
                    setattr(self, f'x{x_index}', x[i].clone())
            self.intrinsic_scatter = nn.Parameter(torch.tensor(row[1]['intrinsic_scatter'], dtype=torch.float64))
        
        def forward(self,i):
            # calculate the model prediction
            # replace x[i] with self.x[i]
            eq = equation
            for x_index in x_indexs:
                eq = eq.replace(f'x{x_index}', f'self.x{x_index}[i]')
            
            # evaluate the equation
            return eval(eq)
        
    model = Model()

    # Define the loss function
    def loglikelihood():
        # term0
        term0 = torch.log(torch.tensor(2 * torch.pi)) * len(df) * (row[1]['unique_number_variables'] + 1)

        #term1
        term1 = (torch.log(M_BH_std_sym**2 + model.intrinsic_scatter**2)).sum()
        for j in range(len(x_indexs)):
            if x_stds[j].sum() != 0:
                term1 += (torch.log(x_stds[j]**2)).sum()

        #term2
        term2 = ((M_BH - model(torch.arange(len(df))))**2 / (M_BH_std_sym**2 + model.intrinsic_scatter**2)).sum()
        
        #term3
        term3 = 0
        for j in range(len(x_indexs)):
            if x_stds[j].sum() != 0:
                term3 += (((x[j] - getattr(model, f'x{x_indexs[j]}')) / x_stds[j])**2).sum()

        return term0 + term1 + term2 + term3
    
    # Define the optimizer
    optimizer = optim.Adam(model.parameters(), lr=0.05)
    scheduler_plateau = optim.lr_scheduler.ReduceLROnPlateau(optimizer,  factor=0.5, patience=10,threshold=1e-6)
    scheduler_linear = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

    # Training loop
    for epoch in range(10000):
    #for epoch in range(1):
        optimizer.zero_grad()
        loss = loglikelihood()
        loss.backward()
        optimizer.step()
        #scheduler_plateau.step(loss)
        scheduler_linear.step()
        if epoch % 100 == 0:
            current_lr = optimizer.param_groups[0]['lr']
            print(f'Epoch {epoch}, LR: {current_lr:.5f}, Loss: {loss.item()}', [getattr(model, f'p{i}').item() for i in range(len(constants))], model.intrinsic_scatter.item())


    # Get the 2nd derivative of the loss function at last epoch
    def compute_hessian(model):
        # Collect initial parameters to get shapes and sizes
        params = list(model.parameters())
        shapes = [p.shape for p in params]
        numels = [p.numel() for p in params]

        # Define a function that takes a flat tensor and returns the loss
        def flat_loss(flat_params):
            # Split the flat tensor into individual parameters
            split_params = []
            offset = 0
            for shape, numel in zip(shapes, numels):
                param = flat_params[offset:offset + numel].view(shape)
                split_params.append(param)
                offset += numel

            # Create a dictionary of parameters for dynamic access
            param_dict = {name: param for name, param in zip(model.state_dict(), split_params)}

            eq = equation
            for x_index in x_indexs:
                eq = eq.replace(f'x{x_index}', f'param_dict["x{x_index}"]')
            for i_index in range(len(constants)):
                eq = eq.replace(f'self.p{i_index}', f'param_dict["p{i_index}"]')
            pred=eval(eq)
            #print(pred)

            # Manually compute the model's output using the split parameters
            # Re-implement the loss calculation using param_dict
            term0 = torch.log(torch.tensor(2 * torch.pi)) * len(df) * (row[1]['unique_number_variables'] + 1)
            
            term1 = (torch.log(M_BH_std_sym**2 + param_dict['intrinsic_scatter']**2)).sum()
            for j in range(len(x_indexs)):
                if x_stds[j].sum() != 0:
                    term1 += (torch.log(x_stds[j]**2)).sum()

            term2 = ((M_BH - pred)**2 / (M_BH_std_sym**2 + param_dict['intrinsic_scatter']**2)).sum()

            term3 = 0
            for j in range(len(x_indexs)):
                if x_stds[j].sum() != 0:
                    term3 += (((x[j] - param_dict[f'x{x_indexs[j]}']) / x_stds[j])**2).sum()
            
            #print(term0 + term1 + term2 + term3)
            return term0 + term1 + term2 + term3


        # Compute Hessian with respect to the flattened initial parameters
        initial_flat = torch.cat([p.detach().flatten() for p in params])
        hessian_matrix = hessian(flat_loss, initial_flat)
        
        return hessian_matrix.numpy()

    hessian_matrix = compute_hessian(model)

    
    # Get cholesky decomposition
    L = np.linalg.cholesky(hessian_matrix)

    if L.diagonal().all() > 0:
        # get log determinant of the hessian with cholesky decomposition
        logdet = 2 * np.sum(np.log(L.diagonal()))

        # inverse of the hessian matrix gives the covariance matrix
        P=np.linalg.inv(L).transpose()
        inv_hessian_matrix = np.dot(P,P.transpose())
        sqrt_cov=np.sqrt(np.diag(inv_hessian_matrix))

    else:
        print('Warning! cholesky decomposition has non-positive diagonal elements')
        logdet = np.nan
        sqrt_cov = np.nan

    # get the string of final fit equation
    optimized_equation = equation
    for i in range(len(constants)):
        optimized_equation = optimized_equation.replace(f'self.p{i}', str(getattr(model, f'p{i}').item()))

      
    # Get the optimized parameters
    optimized_params = {name: param.data for name, param in model.named_parameters()}
    print(optimized_params)

    # residual
    residual_array=((M_BH - model(torch.arange(len(df)))) / torch.sqrt(M_BH_std_sym**2 + model.intrinsic_scatter**2)).detach().numpy()
    
    dim = 1
    for j in range(len(x_indexs)):
        if x_stds[j].sum() != 0:
            residual_array=np.vstack((residual_array,((x[j] - getattr(model, f'x{x_indexs[j]}')) / x_stds[j]).detach().numpy()))
            dim += 1
    
    # test the normality with Anderson of the residuals from a N(0,1) distribution
    p_value=np.zeros(residual_array.shape[0]+1)
    # full flattened residual
    r_data = FloatVector(residual_array.flatten())
    result = r['ad.test'](r_data,null='pnorm',mean=0,sd=1)
    p_value[-1] = result[1][0]
    # residuals for each parameter
    for j in range(residual_array.shape[0]):
        # Convert Python data to R vector
        r_data = FloatVector(residual_array[j])

        # Perform Anderson test (R's Anderson-Darling test)
        result = r['ad.test'](r_data,null='pnorm',mean=0,sd=1)
        p_value[j] = result[1][0]
        


    # BIC
    N_para=np.max([len(constants),len(x_indexs)])+1
    for j in range(len(x_indexs)):
        if x_stds[j].sum() != 0:
            N_para += len(df)
    print('N_para: ',N_para, 'dim: ',dim)
    # BIC = loss.item() + N_para * np.log(len(df)) - N_para * np.log(2 * np.pi)
    BIC = loss.item() + N_para * np.log(len(df))
    BIC_i = loss.item() + N_para * np.log(len(df)) + np.log(logdet) - N_para * np.log(2 * np.pi)

    
    

    


    t_eq.at[row[0],'final_equation'] = optimized_equation
    t_eq.at[row[0],'logdet'] = logdet
    t_eq.at[row[0],'sqrt_cov'] = sqrt_cov
    t_eq.at[row[0],'final_loss'] = loss.item()
    t_eq.at[row[0],'optimized_params'] = {name: param.data for name, param in model.named_parameters()}
    t_eq.at[row[0],'N_para'] = N_para
    t_eq.at[row[0],'residual_array'] = residual_array
    t_eq.at[row[0],'ad_p_value_array'] = p_value
    t_eq.at[row[0],'final_BIC'] = BIC
    t_eq.at[row[0],'final_BIC_i'] = BIC_i
    

  0%|          | 0/791 [00:00<?, ?it/s]

Epoch 0, LR: 0.05000, Loss: -530.4456725753321 [2.5111065715572085, 4.638610785778458] 0.3999327543329131
Epoch 100, LR: 0.05000, Loss: -536.4731692022851 [2.5598253113022857, 4.7348352262692] 0.4189140661486946
Epoch 200, LR: 0.05000, Loss: -536.6170926142161 [2.6720711329182367, 4.783939951629477] 0.41895973039236684
Epoch 300, LR: 0.05000, Loss: -536.6291591699214 [2.7220904399547057, 4.805883488097175] 0.4187050391032983
Epoch 400, LR: 0.05000, Loss: -536.6300033507049 [2.7363888852714955, 4.812155992953553] 0.4186375620892978
Epoch 500, LR: 0.05000, Loss: -536.6300317861334 [2.7391563509652124, 4.813370024346885] 0.4186248348876971
Epoch 600, LR: 0.05000, Loss: -536.6300322435609 [2.7395208122130406, 4.813529905857856] 0.4186231669412669
Epoch 700, LR: 0.05000, Loss: -536.6300322468957 [2.7395527670286146, 4.813543923762469] 0.41862302079133223
Epoch 800, LR: 0.05000, Loss: -536.6300322469057 [2.739554544702601, 4.813544703590593] 0.41862301266131297
Epoch 900, LR: 0.05000, Loss: 

  0%|          | 1/791 [00:08<1:46:36,  8.10s/it]

Epoch 9900, LR: 0.00099, Loss: -536.6300322469056 [2.7395546029483584, 4.813544729141781] 0.41862301239493227
{'p0': tensor(2.7396, dtype=torch.float64), 'p1': tensor(4.8135, dtype=torch.float64), 'x15': tensor([2.4719, 2.5147, 1.5747, 2.3742, 2.2954, 2.2928, 2.1793, 2.3900, 2.3481,
        2.4693, 2.2548, 2.5205, 2.4256, 2.5224, 2.1489, 2.1869, 2.2817, 2.4938,
        2.4154, 2.3154, 2.1346, 2.3065, 2.1579, 2.3739, 2.0174, 2.3308, 2.3461,
        2.2892, 2.3347, 2.4902, 2.3915, 2.4197, 2.2398, 2.4724, 2.4660, 2.0471,
        2.3811, 2.2587, 2.1089, 2.4434, 2.2398, 2.0674, 2.2344, 2.4504, 2.2517,
        2.5097, 2.3522, 2.3984, 2.1936, 2.0499, 2.1482, 2.3575, 2.5194, 2.2181,
        2.0069, 2.1484, 2.5947, 2.3252, 2.4004, 2.0252, 2.5374, 2.2618, 2.3731,
        2.3616, 2.3755, 2.5850, 2.1062, 1.8405, 2.5013, 2.4615, 2.0925, 2.1879,
        1.9960, 2.2903, 2.3073, 2.0294, 2.2833, 2.1391, 2.3652, 2.1807, 2.2150,
        2.1021, 2.0734, 2.0975, 1.9799, 2.1251, 1.9779, 2.0005, 2.2150, 2.35

In [7]:
residual_array.flatten().shape

(188,)

In [8]:
t_eq.iloc[319]

complexity                                                                 5
loss                                                                0.095167
score                                                                0.19008
equation                      ((x15 * 5.2522490187638) - 3.6270176836142696)
sympy_format                        5.2522490187638*x15 - 3.6270176836142696
lambda_format              PySRFunction(X=>5.2522490187638*x15 - 3.627017...
number_constants                                                           2
variables                                                            {'x15'}
number_variables                                                           1
unique_number_variables                                                    1
evolutions                                                                 2
iterations                                                                19
fitting_format                                               p[1]*x15 - p[0]

In [9]:
t_eq.iloc[319]['residual_array'].shape

(2, 94)

In [10]:
t_eq.iloc[319]['ad_p_value_array']

array([9.43559960e-01, 6.38297872e-06, 3.19846589e-06])